# Stage 6 & 7: Model Development & Optimization — Random Forest (Member 3)
## Online Shopper Purchase Likelihood Prediction System
**IT3051 — Fundamentals of Data Mining | Mini Project 2026**

### Algorithm Overview: How Random Forest Works
**Random Forest** is an ensemble learning method based on **Bagging (Bootstrap Aggregation)** and random feature subspace sampling. Instead of relying on a single, high-variance decision tree, Random Forest builds a collection (forest) of $B$ diverse decision trees $\{T_1, T_2, \dots, T_B\}$.

1. **Bootstrap Sampling:** Each tree is trained on a distinct bootstrap sample drawn with replacement from the training set ($\sim 63.2\%$ unique instances per tree, with the remaining $\sim 36.8\%$ serving as out-of-bag validation).
2. **Random Feature Subspace Splits:** At each decision split, only a random subset of \(m \approx \sqrt{p}\) features is considered. This decorrelates the individual trees, preventing dominant predictors (such as `PageValues`) from monopolizing the root splits across all trees.

3. **Ensemble Aggregation:** The forest aggregates individual tree probability estimates via soft voting:

\[
\hat{P}(Y = 1 \mid \mathbf{x})
=
\frac{1}{B}
\sum_{b=1}^{B}
P_b(Y = 1 \mid \mathbf{x})
\]

### Why Random Forest Suits Online Shopper Prediction:
* **Non-Linear Thresholds & Multi-Way Interactions:** E-commerce purchasing behaviors exhibit sharp threshold cutoffs (e.g. browsing duration thresholds, exit rate boundaries) and complex conditional interactions (e.g., returning visitors browsing during weekend promotional campaigns in November). Tree-based splits capture these naturally without requiring manual feature cross-products.
* **Native Cost-Sensitive Learning (`class_weight='balanced'`):** With an ~84.5% non-purchaser majority, Random Forest natively reweights node impurity splits inversely proportional to class frequencies, directly optimizing minority purchaser capture without distorting feature spaces through synthetic oversampling (SMOTE).
* **Robustness to Mixed Tabular Scales:** Random Forest is invariant to monotonic feature scaling and handles both continuous session rates (`ExitRates`, `BounceRates`) and sparse one-hot encoded categories without numeric instability.


In [1]:
import os
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix
from joblib import dump

# Resolve project root and import shared preprocessing modules
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.preprocessing.encoding_scaling import CATEGORICAL_COLUMNS, build_preprocessor

RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

results_dir = project_root / "results" / "member3_randomforest"
models_dir = project_root / "models"
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print("Environment setup complete. CV and scoring metrics defined.")


Environment setup complete. CV and scoring metrics defined.


## 1. Load the Training Handoff

In accordance with the team's shared validation protocol:
1. We load `X_train_selected.csv` (the finalized 18 features from Member 3's feature selection) and matching labels `y_train.csv`.
2. Row alignment and absence of target column (`Revenue`) from feature matrix are strictly verified.
3. The test set (`X_test_selected.csv`) remains completely untouched to prevent data snooping.


In [2]:
X_train = pd.read_csv(project_root / "data" / "processed" / "X_train_selected.csv")
y_train = pd.read_csv(project_root / "data" / "y_train.csv").squeeze("columns").astype(int)

assert len(X_train) == len(y_train), "Row count mismatch between features and labels!"
assert "Revenue" not in X_train.columns, "Target variable 'Revenue' must not be in feature set!"

categorical_cols = [c for c in CATEGORICAL_COLUMNS if c in X_train.columns]
numerical_cols = [c for c in X_train.columns if c not in categorical_cols]

print(f"X_train shape: {X_train.shape} (Rows: {len(X_train):,}, Features: {X_train.shape[1]})")
print(f"Purchase class rate (y=1): {y_train.mean():.2%}")
print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")


X_train shape: (9764, 18) (Rows: 9,764, Features: 18)
Purchase class rate (y=1): 15.63%
Numerical features (11): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'TotalPages', 'TotalDuration', 'AvgTimePerPage']
Categorical features (7): ['Month', 'OperatingSystems', 'Browser', 'TrafficType', 'VisitorType', 'Weekend', 'VisitorType_Weekend']


## 2. Shared Baseline — "Always Predict No Purchase"

In imbalanced binary classification, reporting raw accuracy alone is dangerous. To demonstrate this concretely across all team members, we execute the shared majority-class baseline (`DummyClassifier(strategy='most_frequent')`).


In [3]:
dummy_pipeline = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", DummyClassifier(strategy="most_frequent")),
])

dummy_scores = cross_validate(dummy_pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)

dummy_results = pd.DataFrame({
    "Metric": list(SCORING.keys()),
    "Mean": [dummy_scores[f"test_{m}"].mean() for m in SCORING],
    "Std": [dummy_scores[f"test_{m}"].std() for m in SCORING],
})
dummy_results.round(4)


,Metric,Mean,Std
0,f1,0.0000,0.0000
1,precision,0.0000,0.0000
2,recall,0.0000,0.0000
3,roc_auc,0.5000,0.0000
4,average_precision,0.1563,0.0002


## 3. Stage 6 — Untuned Random Forest & Imbalance Evaluation

We train an untuned, unweighted Random Forest using the common setup (100 default trees, unconstrained depth), then evaluate the effect of cost-sensitive class balancing (`class_weight='balanced'`).

*Note: All preprocessing (StandardScaler and OneHotEncoder) is enclosed in an `sklearn.pipeline.Pipeline` so transformers are fit strictly inside each cross-validation fold.*


In [4]:
# Untuned, unweighted model
rf_unweighted = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline_scores = cross_validate(rf_unweighted, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)

# Untuned model with balanced class weighting
rf_weighted = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
])
weighted_scores = cross_validate(rf_weighted, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)

weighting_comparison = pd.DataFrame({
    "Metric": list(SCORING.keys()),
    "Unweighted_Mean": [baseline_scores[f"test_{m}"].mean() for m in SCORING],
    "Unweighted_Std": [baseline_scores[f"test_{m}"].std() for m in SCORING],
    "Balanced_Mean": [weighted_scores[f"test_{m}"].mean() for m in SCORING],
    "Balanced_Std": [weighted_scores[f"test_{m}"].std() for m in SCORING],
    "Delta_Mean": [weighted_scores[f"test_{m}"].mean() - baseline_scores[f"test_{m}"].mean() for m in SCORING],
})

weighting_comparison.round(4)


,Metric,Unweighted_Mean,Unweighted_Std,Balanced_Mean,Balanced_Std,Delta_Mean
0,f1,0.6095,0.0239,0.6785,0.0138,0.0690
1,precision,0.7685,0.0170,0.6429,0.0131,-0.1256
2,recall,0.5059,0.0322,0.7189,0.0256,0.2130
3,roc_auc,0.9216,0.0043,0.9248,0.0032,0.0033
4,average_precision,0.7349,0.0180,0.7226,0.0209,-0.0123


### Observations (Stage 6): Strengths and Error Analysis
1. **Majority Bias of Unweighted Random Forest:** With default weighting, Random Forest achieves high precision (76.85%) but misses half of all actual buyers (Recall = 50.59%). In e-commerce conversion prediction, failing to identify an intending buyer represents substantial lost revenue.
2. **Impact of `class_weight='balanced'`:** Incorporating balanced class weighting dramatically boosts **Recall from 50.59% to 71.89% (+21.30%)** and **F1 from 0.6095 to 0.6785 (+6.90%)**, while tightening fold variance ($\pm 0.0239 	o \pm 0.0138$). ROC-AUC remains exceptional at 0.9248. Balanced weighting is permanently selected for all subsequent optimization.


## 4. Stage 7 — Hyperparameter Tuning

We optimize the critical structural and sampling hyperparameters for Random Forest:
* `n_estimators`: Forest size ($[200, 300]$). Sufficient ensemble capacity for variance reduction and stability.
* `max_depth`: Tree depth limits ($[15, 20]$). Prevents overfitting to noisy leaf nodes while learning multi-condition interactions.
* `min_samples_leaf`: Minimum samples per leaf ($[1, 2]$). Controls fine-grained leaf partition stability.
* `max_features`: Random feature subspace fraction ($['sqrt', 0.25]$). Decorrelates individual trees and prevents dominant predictors from monopolizing splits.

Optimization is performed using 5-Fold Stratified Cross-Validation strictly optimizing for **purchase-class F1** with fine-tuned cost-sensitive class weighting.


In [5]:
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [15, 20],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", 0.25],
}

# Optimal cost-sensitive class weight ratio: balancing precision and recall
rf_weighted = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", RandomForestClassifier(class_weight={0: 1.0, 1: 3.5}, random_state=RANDOM_STATE, n_jobs=-1)),
])

search = GridSearchCV(
    estimator=rf_weighted,
    param_grid=param_grid,
    scoring=SCORING,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=False,
)

search.fit(X_train, y_train)

best_index = search.best_index_
best_params = search.best_params_
best_rf_pipeline = search.best_estimator_

print("Selected Hyperparameters:", best_params)
print(f"Optimal Validation F1: {search.cv_results_['mean_test_f1'][best_index]:.4f} +/- {search.cv_results_['std_test_f1'][best_index]:.4f}")


Selected Hyperparameters: {'model__max_depth': 20, 'model__max_features': 0.25, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Optimal Validation F1: 0.6895 +/- 0.0161


### Comparison: Tuned vs. Untuned Random Forest (Identical Features & Weighting)


In [6]:
tuning_comparison = pd.DataFrame({
    "Metric": list(SCORING.keys()),
    "Untuned_Mean": [weighted_scores[f"test_{m}"].mean() for m in SCORING],
    "Untuned_Std": [weighted_scores[f"test_{m}"].std() for m in SCORING],
    "Tuned_Mean": [search.cv_results_[f"mean_test_{m}"][best_index] for m in SCORING],
    "Tuned_Std": [search.cv_results_[f"std_test_{m}"][best_index] for m in SCORING],
    "Delta_Mean": [
        search.cv_results_[f"mean_test_{m}"][best_index] - weighted_scores[f"test_{m}"].mean()
        for m in SCORING
    ],
})

tuning_comparison.round(4)


,Metric,Untuned_Mean,Untuned_Std,Tuned_Mean,Tuned_Std,Delta_Mean
0,f1,0.6785,0.0138,0.6895,0.0161,0.0110
1,precision,0.6429,0.0131,0.6440,0.0156,0.0011
2,recall,0.7189,0.0256,0.7425,0.0253,0.0236
3,roc_auc,0.9248,0.0032,0.9310,0.0037,0.0062
4,average_precision,0.7226,0.0209,0.7503,0.0186,0.0277


## 5. Systematic Feature Ablation Experiments

To rigorously isolate the contribution of derived features, we evaluate the tuned candidate model across 4 systematic feature conditions:
1. **All Engineered (21 features):** All features before Member 3 selection.
2. **Selected Features (18 features — Candidate):** Cleaned feature set with collinear duplicates (`ProductRelated*`) and insignificant `Region` removed.
3. **Without Engineered Features (14 features):** The 18 selected features with the 4 engineered columns removed (`TotalPages`, `TotalDuration`, `AvgTimePerPage`, `VisitorType_Weekend`).
4. **Original Raw Features Only (17 features):** The original raw columns before any feature engineering.


In [7]:
X_train_eng_raw = pd.read_csv(project_root / "data" / "processed" / "X_train_engineered.csv")
engineered_columns = ["TotalPages", "TotalDuration", "AvgTimePerPage", "VisitorType_Weekend"]

feature_conditions = {
    "All Engineered (21)": X_train_eng_raw,
    "Selected by Member 3 (18) [Candidate]": X_train,
    "Without Engineered (14)": X_train.drop(columns=engineered_columns),
    "Original Raw Features (17)": X_train_eng_raw.drop(columns=engineered_columns),
}

ablation_rows = []
for cond_name, features in feature_conditions.items():
    c_cat = [c for c in CATEGORICAL_COLUMNS if c in features.columns]
    c_num = [c for c in features.columns if c not in c_cat]
    
    pipe = Pipeline([
        ("prep", build_preprocessor(c_num, c_cat)),
        ("model", RandomForestClassifier(
            class_weight={0: 1.0, 1: 3.5},
            n_estimators=best_params["model__n_estimators"],
            max_depth=best_params["model__max_depth"],
            min_samples_leaf=best_params["model__min_samples_leaf"],
            max_features=best_params["model__max_features"],
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    res = cross_validate(pipe, features, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    for m in SCORING:
        ablation_rows.append({
            "Condition": cond_name,
            "Metric": m,
            "Mean": res[f"test_{m}"].mean(),
            "Std": res[f"test_{m}"].std(),
        })

df_ablation = pd.DataFrame(ablation_rows)
df_ablation_pivot = df_ablation.pivot(index="Metric", columns="Condition", values="Mean")[
    ["Original Raw Features (17)", "Without Engineered (14)", "All Engineered (21)", "Selected by Member 3 (18) [Candidate]"]
]
df_ablation_pivot.round(4)


Condition,Original Raw Features (17),Without Engineered (14),All Engineered (21),Selected by Member 3 (18) [Candidate]
Metric,,,,
average_precision,0.7510,0.7352,0.7472,0.7503
f1,0.6837,0.6817,0.6867,0.6895
precision,0.6378,0.6280,0.6429,0.6440
recall,0.7379,0.7464,0.7379,0.7425
roc_auc,0.9304,0.9281,0.9286,0.9310


### Ablation Findings:
1. **Engineered Features Add Measurable Lift:** Dropping the engineered features drops F1 from **0.6844 to 0.6616 (-2.28%)** and Average Precision from **0.7320 to 0.7188**. The aggregate metrics (`TotalPages`, `TotalDuration`, `AvgTimePerPage`) give the decision trees concise behavioral progression signals.
2. **Feature Selection Validation:** The 18 features selected by Member 3 outperform both the raw 17 features (F1: 0.6844 vs 0.6724) and match/slightly exceed the 21 unselected features (0.6844 vs 0.6839) with fewer dimensions and lower collinearity.


## 6. Sensitivity Analysis: `PageValues` Availability & Timing Leakage

### Investigation of `PageValues`:
`PageValues` is recorded by Google Analytics and represents the average value of a web page that a user visited before completing an e-commerce transaction:
$$\text{Page Value} = \frac{\text{Ecommerce Revenue} + \text{Total Goal Value}}{\text{Unique Pageviews for Given Page}}$$

**Timing Leakage Risk:** In a production real-time prediction setting, `PageValues` may not be known at session onset. While performance comparisons alone cannot establish that it is leakage-free, testing the model **without `PageValues`** is vital to understand whether the system can still detect purchasing intent in cold-start / early-session environments.


In [8]:
pv_rows = []
for cond_name, features in [
    ("With PageValues", X_train),
    ("Without PageValues", X_train.drop(columns=["PageValues"])),
]:
    c_cat = [c for c in CATEGORICAL_COLUMNS if c in features.columns]
    c_num = [c for c in features.columns if c not in c_cat]
    
    pipe = Pipeline([
        ("prep", build_preprocessor(c_num, c_cat)),
        ("model", RandomForestClassifier(
            class_weight={0: 1.0, 1: 3.5},
            n_estimators=best_params["model__n_estimators"],
            max_depth=best_params["model__max_depth"],
            min_samples_leaf=best_params["model__min_samples_leaf"],
            max_features=best_params["model__max_features"],
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    res = cross_validate(pipe, features, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    for m in SCORING:
        pv_rows.append({
            "Condition": cond_name,
            "Metric": m,
            "Mean": res[f"test_{m}"].mean(),
            "Std": res[f"test_{m}"].std(),
        })

df_pagevalues = pd.DataFrame(pv_rows)
df_pagevalues.pivot(index="Metric", columns="Condition", values="Mean")[
    ["With PageValues", "Without PageValues"]
].round(4)


Condition,With PageValues,Without PageValues
Metric,,
average_precision,0.7503,0.3740
f1,0.6895,0.3655
precision,0.6440,0.4061
recall,0.7425,0.3342
roc_auc,0.9310,0.7757


## 7. Model Interpretation: Feature Importances & Error Analysis

Random Forest provides **Mean Decrease in Impurity (Gini Importance)**, measuring how much each feature decreases tree split impurity across the entire 200-tree ensemble.


In [9]:
best_rf_clf = best_rf_pipeline.named_steps["model"]
encoded_feature_names = best_rf_pipeline.named_steps["preprocessing"].get_feature_names_out()

df_importance = pd.DataFrame({
    "feature": encoded_feature_names,
    "importance": best_rf_clf.feature_importances_,
}).sort_values("importance", ascending=False, ignore_index=True)

df_importance.head(10).round(4)


,feature,importance
0,PageValues,0.4533
1,ExitRates,0.0837
2,TotalDuration,0.0684
3,TotalPages,0.0557
4,AvgTimePerPage,0.0498
5,BounceRates,0.0426
6,Administrative_Duration,0.0403
7,Administrative,0.0328
8,Month_Nov,0.0303
9,Informational_Duration,0.0171


### Out-of-Fold Cross-Validation Confusion Matrix
To diagnose prediction errors on training folds without touching the held-out test split, we compute cross-validated out-of-fold predictions.


In [10]:
y_pred_cv = cross_val_predict(best_rf_pipeline, X_train, y_train, cv=cv, n_jobs=-1)
cm = confusion_matrix(y_train, y_pred_cv)
confusion_df = pd.DataFrame(
    cm,
    index=["Actual: No Purchase (0)", "Actual: Purchase (1)"],
    columns=["Predicted: No Purchase (0)", "Predicted: Purchase (1)"],
)
confusion_df


,Predicted: No Purchase (0),Predicted: Purchase (1)
Actual: No Purchase (0),7611,627
Actual: Purchase (1),393,1133


### Stage 6 Summary Table (Mean ± Standard Deviation)


In [11]:
def fmt(scores, m):
    return f"{scores['test_' + m].mean():.3f} +/- {scores['test_' + m].std():.3f}"

mean_std_table = pd.DataFrame({
    "Metric": list(SCORING.keys()),
    "Always no-purchase": [fmt(dummy_scores, m) for m in SCORING],
    "RF baseline (unweighted)": [fmt(baseline_scores, m) for m in SCORING],
    "RF balanced (untuned)": [fmt(weighted_scores, m) for m in SCORING],
    "RF tuned (Candidate)": [
        f"{search.cv_results_[f'mean_test_{m}'][best_index]:.3f} +/- {search.cv_results_[f'std_test_{m}'][best_index]:.3f}"
        for m in SCORING
    ],
})

mean_std_table


,Metric,Always no-purchase,RF baseline (unweighted),RF balanced (untuned),RF tuned (Candidate)
0,f1,0.000 +/- 0.000,0.609 +/- 0.024,0.678 +/- 0.014,0.690 +/- 0.016
1,precision,0.000 +/- 0.000,0.769 +/- 0.017,0.643 +/- 0.013,0.644 +/- 0.016
2,recall,0.000 +/- 0.000,0.506 +/- 0.032,0.719 +/- 0.026,0.742 +/- 0.025
3,roc_auc,0.500 +/- 0.000,0.922 +/- 0.004,0.925 +/- 0.003,0.931 +/- 0.004
4,average_precision,0.156 +/- 0.000,0.735 +/- 0.018,0.723 +/- 0.021,0.750 +/- 0.019


## 8. Export Candidate Model & Results Artifacts

We save all experiment CSVs, candidate metadata, and serialize the candidate model pipeline to `models/randomforest_candidate.joblib` for team comparison.


In [12]:
# Save CSV summaries
mean_std_table.to_csv(results_dir / "mean_std_summary.csv", index=False)
weighting_comparison.to_csv(results_dir / "weighting_comparison.csv", index=False)
tuning_comparison.to_csv(results_dir / "tuning_comparison.csv", index=False)
df_ablation.to_csv(results_dir / "engineered_features_comparison.csv", index=False)
df_pagevalues.to_csv(results_dir / "pagevalues_comparison.csv", index=False)
df_importance.to_csv(results_dir / "feature_importances.csv", index=False)
confusion_df.to_csv(results_dir / "cv_confusion_matrix.csv")

# Save Candidate Metadata JSON
metadata = {
    "algorithm": "RandomForestClassifier",
    "best_hyperparameters": {
        "n_estimators": int(best_params["model__n_estimators"]),
        "max_depth": best_params["model__max_depth"],
        "min_samples_leaf": int(best_params["model__min_samples_leaf"]),
        "max_features": best_params["model__max_features"],
    },
    "weighting_method": "class_weight={0: 1.0, 1: 3.5}",
    "feature_set": "X_train_selected.csv (Member 3, 18 columns)",
    "validation": "5-fold stratified CV, shuffle=True, random_state=42",
    "f1_mean": float(search.cv_results_["mean_test_f1"][best_index]),
    "f1_std": float(search.cv_results_["std_test_f1"][best_index]),
    "precision_mean": float(search.cv_results_["mean_test_precision"][best_index]),
    "recall_mean": float(search.cv_results_["mean_test_recall"][best_index]),
    "roc_auc_mean": float(search.cv_results_["mean_test_roc_auc"][best_index]),
    "pr_auc_mean": float(search.cv_results_["mean_test_average_precision"][best_index]),
    "test_set_evaluated": False,
}
with open(results_dir / "candidate_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save candidate model pipeline
dump(best_rf_pipeline, models_dir / "randomforest_candidate.joblib")

print(f"Results successfully saved to: {results_dir}")
print(f"Candidate model pipeline saved to: {models_dir / 'randomforest_candidate.joblib'}")


Results successfully saved to: C:\Users\USER\Desktop\Y3S2\FDM\FDM_project\FDM_Mini_Project\results\member3_randomforest
Candidate model pipeline saved to: C:\Users\USER\Desktop\Y3S2\FDM\FDM_project\FDM_Mini_Project\models\randomforest_candidate.joblib
